In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/data'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
import re
from sklearn.utils import shuffle
# --- 1. LOAD BENGALI DATA ---
path_bengali = "data/BSMDD_v3_textcleaned.csv"
df_bengali = pd.read_csv(path_bengali)
df_bengali = df_bengali[['text', 'label']].dropna()
# Ensure text is string
df_bengali['text'] = df_bengali['text'].astype(str)

# ---------change for reproducing purposes----------
# --- 2. LOAD TWEET DATA (Multilingual) ---
# path_tweets = "/kaggle/input/depression-tweets/master_dataset_7z.csv"
# df_tweets = pd.read_csv(path_tweets, usecols=['text', 'label'])
# df_tweets = df_tweets.dropna()

# --- 3. CLEAN TWEETS ---
def clean_multilingual_tweets(text):
    text = str(text)
    # Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # Remove mentions
    text = re.sub(r'@\S+', '', text)
    # Handle [SEP]
    text = text.replace('[SEP]', ' ')
    # Remove HTML
    text = re.sub(r'<.*?>', '', text)
    # Remove new lines
    text = text.replace('\n', ' ')
    return text.strip()

# print("Cleaning tweets...")
# df_tweets['text'] = df_tweets['text'].apply(clean_multilingual_tweets)

# --- 4. COMBINE & SHUFFLE ---
# ---------change for reproducing purposes----------
df_combined = df_bengali #pd.concat([df_bengali, df_tweets], ignore_index=True)
df_combined = shuffle(df_combined, random_state=42).reset_index(drop=True)
# df_combined = df_bengali
print(f"Total Combined Samples: {len(df_combined)}")
print(df_combined.head())

Total Combined Samples: 21910
                                                text  label
0  ওয়াইফাই ছোট গল্প কুকুরটিকে বেড়াতে যেতাম লোকে...      0
1  বর্তমানে ইউনিতে কোভিডের কারণে বন্ধুত্ব পারিনি ...      1
2  টিকটক ততটাই খারাপ ভাবতে হ্যাঁ সপ্তাহ টিকটক ইনস...      0
3  মি ভাল আসলে নিজেকে হত্যা পারিনি সম্ভব পান করেছ...      1
4  চারপাশে বর্ণবাদী সংস্কৃতি পরিবার কথা বলছিলাম ব...      0


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
from sklearn.model_selection import train_test_split

# ---------change for reproducing purposes----------
df_combined = df_combined.sample(frac=0.02, random_state=42)

X = df_combined['text'].values
y = df_combined['label'].values

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
MAX_LEN = 128
BATCH_SIZE = 32
# Check how many of your texts are longer than MAX_LEN
lengths = [len(text.split()) for text in X_train]
over_limit = sum(1 for x in lengths if x > MAX_LEN)

print(f"Total samples: {len(lengths)}")
print(f"Samples being truncated: {over_limit} ({over_limit/len(lengths)*100:.2f}%)")
print(f"Average length: {np.mean(lengths)}")

Total samples: 350
Samples being truncated: 52 (14.86%)
Average length: 77.93714285714286


In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
# !pip install -U transformers safetensors
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, LSTM, Bidirectional, Conv1D, MaxPooling1D, TimeDistributed, GlobalMaxPooling1D, Flatten
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from transformers import AutoTokenizer, TFAutoModel
from sklearn.model_selection import train_test_split

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
MAX_LEN = 128       # Size of each window
MAX_CHUNKS = 8      # Number of windows per text (8 * 128 = 1024 tokens)
BATCH_SIZE = 4      # Small batch size because Sliding Window uses a lot of memory
MODEL_NAME = "data/models/LaBSE"


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
def sliding_window_encode(texts, tokenizer, max_len=MAX_LEN, max_chunks=MAX_CHUNKS):
    # Total tokens we need to fill 8 windows (8 * 128 = 1024)
    total_needed = max_len * max_chunks
    
    print(f"Tokenizing {len(texts)} samples...")
    
    # 1. Tokenize everything at once up to our total limit (1024)
    # This stops the 512 warning because we are explicitly truncating at 1024
    encodings = tokenizer(
        texts.tolist(),
        add_special_tokens=True,
        max_length=total_needed,
        padding='max_length',
        truncation=True,
        return_tensors="np"
    )
    
    input_ids = encodings['input_ids']
    attention_masks = encodings['attention_mask']
    
    # 2. Reshape the flat (Samples, 1024) into (Samples, 8, 128)
    # This automatically chops the 1024 tokens into 8 windows of 128
    transformed_ids = input_ids.reshape(-1, max_chunks, max_len)
    transformed_masks = attention_masks.reshape(-1, max_chunks, max_len)
    
    return transformed_ids, transformed_masks

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# Assuming you have X_train, X_test from your split
print("Starting Sliding Window encoding...")

train_ids, train_masks = sliding_window_encode(X_train, tokenizer)
test_ids, test_masks = sliding_window_encode(X_test, tokenizer)

print(f"New Shape of train_ids: {train_ids.shape}") 
# Result should be: (Samples, 8, 128)
print("Data is now ready for the model.")

Starting Sliding Window encoding...
Tokenizing 350 samples...
Tokenizing 88 samples...
New Shape of train_ids: (350, 8, 128)
Data is now ready for the model.


In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
import tensorflow.keras.backend as K
from tensorflow.keras.layers import Layer

class Attention(Layer):
    def __init__(self, **kwargs):
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name='attention_weight', shape=(input_shape[-1], 1), 
                                 initializer='normal', trainable=True)
        self.b = self.add_weight(name='attention_bias', shape=(input_shape[1], 1), 
                                 initializer='zeros', trainable=True)        
        super(Attention, self).build(input_shape)

    def call(self, x):
        e = K.tanh(K.dot(x, self.W) + self.b)
        a = K.softmax(e, axis=1)
        output = x * a
        return K.sum(output, axis=1)

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
import keras
from keras import ops

# We wrap the LaBSE + CNN logic into a custom layer to avoid Keras 3 "Symbolic" errors
class LaBSEEmbeddingLayer(keras.layers.Layer):
    def __init__(self, model_name, **kwargs):
        super().__init__(**kwargs)
        # Load the model inside the layer
        self.labse_model = TFAutoModel.from_pretrained(model_name, from_pt=True)
        self.labse_model.trainable = False
        
        # Define CNN layers inside this wrapper
        # ---------change for reproducing purposes----------
        self.cnn = Conv1D(64, kernel_size=3, padding='same', activation='relu') # 128
        self.pool = MaxPooling1D(pool_size=2)
        self.global_pool = GlobalMaxPooling1D()

    def call(self, inputs):
        input_ids, input_mask = inputs
        
        # 1. Flatten windows: (Batch, 8, 128) -> (Batch * 8, 128)
        # We use keras.ops.reshape for compatibility
        flat_ids = ops.reshape(input_ids, (-1, MAX_LEN))
        flat_mask = ops.reshape(input_mask, (-1, MAX_LEN))

        # 2. Get LaBSE Embeddings
        embeddings = self.labse_model(flat_ids, attention_mask=flat_mask)[0]

        # 3. Apply CNN
        x = self.cnn(embeddings)
        x = self.pool(x)
        x = self.global_pool(x) # Result is (Batch * 8, 128)

        # 4. Reshape back for LSTM: (Batch * 8, 128) -> (Batch, 8, 128)
        # We use -1 for the batch dimension
        output = ops.reshape(x, (-1, MAX_CHUNKS, 128))
        return output

def build_sliding_window_model():
    # Define Inputs
    input_ids = Input(shape=(MAX_CHUNKS, MAX_LEN), dtype=tf.int32, name="input_ids")
    input_mask = Input(shape=(MAX_CHUNKS, MAX_LEN), dtype=tf.int32, name="attention_mask")

    # 1. Use the Custom Wrapper Layer
    # This handles the LaBSE call and the CNN in one block
    window_features = LaBSEEmbeddingLayer(MODEL_NAME)([input_ids, input_mask])

    # 2. Bi-LSTM (Processing the sequence of 8 windows)
    # ---------change for reproducing purposes----------
    x = Bidirectional(LSTM(32, return_sequences=True))(window_features) # 64
    
    # 3. Attention (Highlighting the most depressive windows)
    x = Attention()(x)

    # 4. Final Classification Layers
    # ---------change for reproducing purposes----------
    x = Dense(32, activation='relu')(x) # 64
    x = Dropout(0.4)(x)
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=[input_ids, input_mask], outputs=outputs)
    return model

# Re-instantiate the model
model = build_sliding_window_model()
model.summary()

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['embeddings.position_ids']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertModel for predictions without further training.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_ids (InputLayer)    │ (None, 8, 128)         │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention_mask            │ (None, 8, 128)         │              0 │ -                      │
│ (InputLayer)              │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ la_bse_embedding_layer    │ (None, 8, 128)         │        147,520 │ input_ids[0][0],       │
│ (LaBSEEmbeddingLayer)     │                        │                │ attention_mask[0][0]   │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ bidirectional             │ (None, 8, 64)          │         41,216 │ la_bse_embedding_laye… │
│ (Bidirectional)           │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ attention (Attention)     │ (None, 64)             │             72 │ bidirectional[0][0]    │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense (Dense)             │ (None, 32)             │          2,080 │ attention[0][0]        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dropout (Dropout)         │ (None, 32)             │              0 │ dense[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ dense_1 (Dense)           │ (None, 1)              │             33 │ dropout[0][0]          │
└───────────────────────────┴────────────────────────┴────────────────┴────────────────────────┘

 Total params: 190,921 (745.79 KB)

 Trainable params: 190,921 (745.79 KB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# --- [CELL 10]: ---
# cell_state: edited
# execution_status: {'status': 'error', 'done': True, 'execution_count': 11}
# === BEFORE (original) ===
# optimizer = Adam(learning_rate=1e-4)
# model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
# 
# callbacks = [
#     EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
#     ModelCheckpoint('sliding_window_model.keras', save_best_only=True)
# ]
# 
# print("Starting training (Using Sliding Windows)...")
# history = model.fit(
#     [train_ids, train_masks], y_train,
#     validation_data=([test_ids, test_masks], y_test),
#     epochs=2, #15,
#     batch_size=BATCH_SIZE, # 4
#     callbacks=callbacks
# )

# === AFTER (edited) ===
from sklearn.model_selection import train_test_split

# Resample and split the data fresh to ensure X and y are aligned
df_sampled = df_combined.sample(frac=0.02, random_state=42)

X = df_sampled['text'].values
y = df_sampled['label'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Verify the shapes match
print(f"X_train samples: {len(X_train)}")
print(f"y_train samples: {len(y_train)}")
print(f"X_test samples: {len(X_test)}")
print(f"y_test samples: {len(y_test)}")

# Re-encode the data with the fresh splits
print("Encoding training data...")
train_ids, train_masks = sliding_window_encode(X_train, tokenizer)
print("Encoding test data...")
test_ids, test_masks = sliding_window_encode(X_test, tokenizer)

print(f"train_ids shape: {train_ids.shape}")
print(f"train_masks shape: {train_masks.shape}")

print(f"\ny_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Verify batch alignment
num_batches_train = len(train_ids) // BATCH_SIZE
num_batches_test = len(test_ids) // BATCH_SIZE
print(f"\nComplete batches in train set: {num_batches_train}")
print(f"Complete batches in test set: {num_batches_test}")

print("\nStarting training (Using Sliding Windows)...")
history = model.fit(
    [train_ids, train_masks], y_train,
    validation_data=([test_ids, test_masks], y_test),
    epochs=2,
    batch_size=BATCH_SIZE,
    callbacks=callbacks
)

X_train samples: 7
y_train samples: 7
X_test samples: 2
y_test samples: 2
Encoding training data...
Tokenizing 7 samples...
Encoding test data...
Tokenizing 2 samples...
train_ids shape: (7, 8, 128)
train_masks shape: (7, 8, 128)

y_train shape: (7,)
y_test shape: (2,)

Complete batches in train set: 1
Complete batches in test set: 0

Starting training (Using Sliding Windows)...


NameError: name 'callbacks' is not defined

In [12]:
# 1. LOAD MODEL & CHECK FIX IS IN PLACE
from tensorflow.keras.models import load_model

try:
    model = load_model('sliding_window_model.keras', 
                       custom_objects={'Attention': Attention, 'LaBSEEmbeddingLayer': LaBSEEmbeddingLayer})
except Exception as e:
    raise AssertionError(f"Model loading failed (likely reshape dimension mismatch): {e}")

# 2. VALIDATE TRAINING COMPLETION
assert 'history' in globals(), 'Training did not complete - history missing'
assert 'loss' in history.history and len(history.history['loss']) > 0, 'No training loss recorded'
assert 'accuracy' in history.history and len(history.history['accuracy']) > 0, 'No accuracy recorded'
print(f"✓ Training completed: {len(history.history['loss'])} epochs recorded")

# 3. VALIDATE LAYER ARCHITECTURE & FIX
# Build a standalone test of the embedding layer
test_layer = LaBSEEmbeddingLayer(MODEL_NAME)
test_ids_sample = train_ids[:2]  # (2, 8, 128)
test_mask_sample = train_masks[:2]  # (2, 8, 128)

embedding_output = test_layer([test_ids_sample, test_mask_sample])
expected_shape = (2, MAX_CHUNKS, 64)  # Fix ensures output is (batch, 8, 64)

assert embedding_output.shape == expected_shape, (
    f"LaBSEEmbeddingLayer output shape {embedding_output.shape} != expected {expected_shape}. "
    f"Fix not applied: reshape must use 64, not 128"
)

# 4. VALIDATE END-TO-END MODEL OUTPUTS
batch_n = min(BATCH_SIZE, len(y_test))
predictions = model.predict([test_ids[:batch_n], test_masks[:batch_n]], verbose=0)

assert predictions.shape == (batch_n, 1), (
    f"Model output shape {predictions.shape} != expected ({batch_n}, 1)"
)
assert np.all((predictions >= 0) & (predictions <= 1)), "Predictions outside [0,1] range"

# 5. CATCH DELETION/NO-OP PATCHES
# If reshape was commented out or deleted, model would have been created but would fail here
assert np.mean(predictions) > 0 and np.mean(predictions) < 1, (
    "Model predictions are degenerate (all 0 or all 1) - likely reshape was deleted/not applied"
)

# 6. VALIDATE FULL BATCH INFERENCE
y_pred_prob = model.predict([test_ids, test_masks], verbose=0)
y_pred = (y_pred_prob > 0.5).astype(int)

from sklearn.metrics import accuracy_score, classification_report, f1_score
acc = accuracy_score(y_test, y_pred)
macro_f1 = f1_score(y_test, y_pred, average='macro')

assert 0 <= acc <= 1, f"Accuracy {acc} out of valid range"
assert 0 <= macro_f1 <= 1, f"F1 score {macro_f1} out of valid range"

print(classification_report(y_test, y_pred, digits=4))

# 7. CATCH DIMENSION VARIANTS & PLAUSIBLE WRONG PATCHES
# Verify the critical fix: if CNN output is 64 but reshape tries 128, model load/predict fails
# Verify consistency: whatever CNN outputs, reshape must match
test_all_data = model.predict([test_ids, test_masks], verbose=0)
assert test_all_data.shape == (len(test_ids), 1), (
    f"Batch prediction shape inconsistent: {test_all_data.shape} expected ({len(test_ids)}, 1)"
)
print(f"All test cases passed")

AssertionError: Model loading failed (likely reshape dimension mismatch): File not found: filepath=sliding_window_model.keras. Please ensure the file is an accessible `.keras` zip file.